<a href="https://colab.research.google.com/github/penguinmuffin/cac_2025/blob/main/cac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyAudioAnalysis
!pip install mediapipe opencv-python
!pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 17.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyAudioAnalysis: filename=pyAudioAnalysis-0.3.14-py3-none-any.whl size=41264371 sha256=3591ff62cdb0b552a6df5b530aa43d8ca51629f92780149376b51fb6d630ffa4
  Stored in directory: /root/.cache/pip/wheels/24/9e/ab/d18e8a5866e4b4edfc012d6576e5ed699dd2aa4acbab8e8c90
Successfully built pyAudioAnalysis
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [2]:
import whisper
from pyAudioAnalysis import ShortTermFeatures
import wave
import numpy as np
import cv2
import mediapipe as mp

In [3]:
def analyze_speech_whisper(audio_file):
    model = whisper.load_model("small")
    result = model.transcribe(audio_file)

    transcript = result["text"].lower()
    filler_words = ["um", "uh", "like", "you know", "so"]
    found_fillers = [word for word in filler_words if word in transcript]

    return {
        "transcript": result["text"],
        "filler_words": found_fillers,
        "num_fillers": len(found_fillers)
    }


In [4]:
def analyze_audio_features(audio_file):
    with wave.open(audio_file, "rb") as wf:
        framerate = wf.getframerate()
        n_frames = wf.getnframes()
        signal = np.frombuffer(wf.readframes(n_frames), dtype=np.int16)

    features, feature_names = ShortTermFeatures.feature_extraction(
        signal, framerate,
        0.05 * framerate,  # window size
        0.025 * framerate  # step size
    )

    avg_pitch = np.mean(features[0])   # ZCR (proxy for pitch variability)
    avg_energy = np.mean(features[1])  # Energy
    speech_rate = len(signal) / framerate / (len(features[0]) * 0.025)

    return {
        "avg_pitch_proxy": float(avg_pitch),
        "avg_energy": float(avg_energy),
        "speech_rate": float(speech_rate)
    }


In [5]:
def analyze_video_features(video_file):
    mp_face_mesh = mp.solutions.face_mesh
    mp_pose = mp.solutions.pose

    cap = cv2.VideoCapture(video_file)
    eye_contact_score = []
    posture_score = []

    with mp_face_mesh.FaceMesh(refine_landmarks=True) as face_mesh, \
         mp_pose.Pose() as pose:

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Face landmarks
            face_results = face_mesh.process(rgb_frame)
            if face_results.multi_face_landmarks:
                for face_landmarks in face_results.multi_face_landmarks:
                    left_eye = face_landmarks.landmark[33]
                    right_eye = face_landmarks.landmark[263]
                    # Approximate "eye contact" if eyes near center
                    center_offset = abs((left_eye.x + right_eye.x) / 2 - 0.5)
                    eye_contact_score.append(1 - center_offset)

            # Posture landmarks
            pose_results = pose.process(rgb_frame)
            if pose_results.pose_landmarks:
                left_shoulder = pose_results.pose_landmarks.landmark[11]
                right_shoulder = pose_results.pose_landmarks.landmark[12]
                shoulder_diff = abs(left_shoulder.y - right_shoulder.y)
                posture_score.append(1 - shoulder_diff)  # smaller diff = better posture

    cap.release()

    return {
        "eye_contact": float(np.mean(eye_contact_score)) if eye_contact_score else 0,
        "posture": float(np.mean(posture_score)) if posture_score else 0
    }

In [6]:
def run_full_analysis(audio_file, video_file):
    speech_results = analyze_speech_whisper(audio_file)
    audio_results = analyze_audio_features(audio_file)
    video_results = analyze_video_features(video_file)

    return {
        "speech": speech_results,
        "audio": audio_results,
        "video": video_results
    }
    results["scores"] = score_presentation(results)
    return results

In [ ]:
def score_presentation(results):
    # Extract values
    num_fillers = results["speech"]["num_fillers"]
    avg_pitch = results["audio"]["avg_pitch_proxy"]
    avg_energy = results["audio"]["avg_energy"]
    speech_rate = results["audio"]["speech_rate"]
    eye_contact = results["video"]["eye_contact"]
    posture = results["video"]["posture"]

    # --- Clarity ---
    # High fillers = lower clarity (scaled down)
    clarity = max(0, 1 - (num_fillers * 0.05))

    # --- Confidence ---
    # Confidence = posture + pitch variation (not too monotone)
    confidence = (posture + min(avg_pitch * 10, 1.0)) / 2

    # --- Engagement ---
    # Engagement = eye contact + moderate speech rate + energy
    # Ideal speech rate ~ 3–6 words/sec
    rate_score = 1 - abs(speech_rate - 4.5) / 4.5
    rate_score = max(0, min(1, rate_score))
    energy_score = min(avg_energy / 1000, 1.0)  # normalize energy
    engagement = (eye_contact + rate_score + energy_score) / 3

    return {
        "clarity": round(float(clarity), 2),
        "confidence": round(float(confidence), 2),
        "engagement": round(float(engagement), 2)
    }

In [ ]:
if __name__ == "__main__":
    results = run_full_analysis("sample_presentation.wav", "sample_presentation.mp4")
    print(results)